In [16]:
# Import ESM classes from HF transformers
from transformers import EsmTokenizer, EsmForMaskedLM, AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
import torch.nn.functional as F
import random
import wandb

In [1]:
# Define path to uniref50 fasta
uniref50 = '/home/ubuntu/uniref50_subset.fasta'
# Define path to ESM model #8M parameter model

# Create a dataset class for training that comes from the uniref50 fasta
class UniRef50Dataset(Dataset):
    # Inputs are a path to the uniref50 fasta sequences, the ESM tokenizer name, and a max length of sequence cutoff
    def __init__(self, fasta_path, tokenizer_name, max_length = 1024):
        self.sequences = self.read_fasta(fasta_path)
        self.tokenizer = EsmTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length

    
    def read_fasta(self, fasta_path):
        ''' Get list of sequences from the fasta file '''
        sequences = []
        seq = []
        with open(fasta_path) as f:
            for line in f:
                line = line.strip()
                if line.startswith(">"):
                    if seq:
                        sequences.append("".join(seq))
                        seq = []
                else:
                    seq.append(line)
            if seq:
                sequences.append("".join(seq))
                
            return sequences
            
    def __len__(self):
        ''' Returns number of sequences in the dataset '''
        return len(self.sequences)

    def __getitem__(self, idx):
    """
    Retrieve and tokenize a protein sequence.

    Parameters
    ----------
    idx : int
        Index of the sequence to retrieve.

    Returns
    -------
    dict
        A dictionary containing:
        
        - **input_ids** (torch.Tensor): Tokenized sequence IDs of shape `(L,)`,
          where `L <= max_length`.
        - **attention_mask** (torch.Tensor): Attention mask of shape `(L,)`
          indicating which tokens are padding (0) vs. real sequence (1).
    """
        seq = self.sequences[idx]

        # Create circular permutation of the sequence
        i = random.randint(0, len(seq)-1)
        seq_cp = seq[i:] + seq[:i]

        #Tokenize the circular permutation
        tokenized = self.tokenizer(
            seq_cp,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {key: val.squeeze(0) for key, val in tokenized.items()}
 

IndentationError: expected an indented block (949120645.py, line 54)

In [30]:
def generate_dataset_splits(dataset, splits = [0.70,0.15,0.15]):
    '''
    Create dataset splits
    '''
    dataset_size = len(dataset)
    train_size = int(dataset_size*splits[0])
    val_size = int(dataset_size*splits[1])
    test_size = dataset_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

    return train_dataset, val_dataset, test_dataset

def generate_dataloader(train_dataset, val_dataset, test_dataset, Batch_size):
    '''
    Generate dataloaders for dataset splits
    '''
    train_loader = DataLoader(train_dataset, batch_size=Batch_size, shuffle=True, pin_memory=True, prefetch_factor=2, num_workers=16) #pin_memory=True, prefetch_factor=2, num_workers=16 #load ahead
    val_loader = DataLoader(val_dataset, batch_size=Batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=Batch_size, shuffle=False)
    
    
    return train_loader, val_loader, test_loader

def generate_training_mask(input_ids, mask_prob = 0.15):
    '''
    Generate masked_input_ids
    input: input_ids
    output: input_ids with random 15% tokens set to masked and non-masked tokens set to -100 so they're ignored during loss calculation
    '''
    # Create set of special ids from the tokenizer which we'll not want to mask
    special_ids = {
    tokenizer.cls_token_id,
    tokenizer.pad_token_id,
    tokenizer.eos_token_id,
    tokenizer.mask_token_id,
    tokenizer.unk_token_id
}   
    print('ids', input_ids)
    # copy the input ids to a masked inputs ids tensor
    masked_input_ids = input_ids.clone()

    # Create a labels tensor used for ground truth labels during training
    ground_truth_labels = input_ids.clone()
    
    # assign random number 0 to 1 to each input id
    rand = torch.rand(input_ids.shape).to(device)
    # Create a mask; all positions less than 0.15 (mask prob) and do not mask the special tokens
    mask_arr = (rand < mask_prob) & (~input_ids.isin(torch.tensor(list(special_ids), device=input_ids.device)))
    # Apply the mask to the labels; set the masked positions equal to the tokenizer mask token id
    masked_input_ids[mask_arr] = tokenizer.mask_token_id
    # For the labels, set the tokens that are unmasked to -100 which means these tokens will not contribute to the calculation of the loss function
    ground_truth_labels[~mask_arr] = -100

    return masked_input_ids, ground_truth_labels

In [19]:
    tokenizer = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")


In [20]:
special_ids = {
    tokenizer.cls_token_id,
    tokenizer.pad_token_id,
    tokenizer.eos_token_id,
    tokenizer.mask_token_id,
    tokenizer.unk_token_id
}   

for i in special_ids:
    print(i)
type(tokenizer.mask_token_id)

0
1
2
32
3


int

In [21]:
tokenizer = train_loader.dataset.dataset.tokenizer 

NameError: name 'train_loader' is not defined

In [22]:
def setup_model(model_name="facebook/esm2_t6_8M_UR50D"):
    """
    Initialize model for fine-tuning
    """
    #tokenizer = EsmTokenizer.from_pretrained(model_name)
    model = EsmForMaskedLM.from_pretrained(model_name)
    
    return model

In [23]:
def wandb_run(run_name):
    run = wandb.init(
        entity="aidenkzj",  # Change to your W&B entity
        project="esm-circular-permutation-finetune",  # Change to your project name
        name=run_name,  # Set run name dynamically
        config={
            "learning_rate": 2e-5,
            "architecture": "ESM-MLM",
            "dataset": "circularly-permuted-sequences",
            "epochs": 3,
            "mask_prob": 0.15,
            "batch_size": 64,  # adjust based on your DataLoader
            "optimizer": "AdamW",
            "loss_fn": "CrossEntropyLoss(ignore_index=-100)",
        },
    )
    return run


In [36]:
def training_loop(model, train_loader, val_loader, num_epochs=10, lr=1e-4):
    """
    Training loop for fine-tuning ESM on circularly-permuted sequences
    """
    # Move model to the GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Define optimizer; give the optimizer the model parameters and learning rate as inputs. 
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # Integrate W&B
    run = wandb_run("esm_cp_ft_test_run_001")

    # Train for num_epochs
    for epoch in range(num_epochs):
        # Set model to train and loss to zero
        model.train()
        total_loss = 0
        accumulation_steps = 8
        optimizer.zero_grad()
        # Loop through the batches in the dataloader
        for step, batch in enumerate(train_loader):
            
            # Move the inputs from the batch to the GPU
            #print(batch)
            #break
            input_ids_tokenized = batch['input_ids'].to(device)
            print(input_ids_tokenized.shape)
            break
            # Generate masked inputs and ground truth labels
            masked_input_ids, ground_truth_labels = generate_training_mask(input_ids_tokenized, mask_prob=0.15)
            # Get the attention mask (masks out padding tokens during self-attention
            break
            attention_mask = batch['attention_mask'].to(device)
            # Generate labels using mask function
            #labels = batch['labels'].to(device) # Loader doesn't have labels in this case

            # Get outputs from the model
            outputs = model(
                input_ids=masked_input_ids,
                attention_mask=attention_mask,
                labels=ground_truth_labels
            )

            # Scale loss for gradient accumulation
            loss = outputs.loss / accumulation_steps
            loss.backward()
    
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
            # Step optimizer every accumulation_steps
            if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
                optimizer.step()
    
            total_loss += loss.item() * accumulation_steps  # undo scaling for logging

        
        avg_loss = total_loss / len(train_loader)
        run.log({"train_loss": avg_loss, "epoch": epoch + 1})
        #print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")
        
        # Validation
        model.eval()
        
        val_total_loss = 0

        with torch.no_grad():
            for batch in val_loader:
                # Move batch to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
        
                # Mask for MLM (same as training)
                masked_input_ids, ground_truth_labels = generate_training_mask(input_ids, mask_prob=0.15)
        
                # Forward pass
                outputs = model(
                    input_ids=masked_input_ids,
                    attention_mask=attention_mask,
                    labels=ground_truth_labels
                )
        
                # Collect loss
                val_total_loss += outputs.loss.item()
        
        # Average validation loss for the epoch
        avg_val_loss = val_total_loss / len(val_loader)
        run.log({"val_loss": avg_val_loss, "epoch": epoch + 1})
        #print(f"Validation Loss: {avg_val_loss:.4f}")
        

In [37]:
def main():
    print('> Loading Model, Tokenizer:')
    model = setup_model()
    print('> Loading Dataset:')
    dataset = UniRef50Dataset(
    fasta_path="/home/ubuntu/uniref50_subset.fasta",
    tokenizer_name="facebook/esm2_t6_8M_UR50D",
    max_length=1024
)
    print('> Loading Dataset splits:')
    train_dataset, val_dataset, test_dataset = generate_dataset_splits(dataset)
    print('> Loading DataLoaders:')
    train_loader, val_loader, test_loader = generate_dataloader(train_dataset, val_dataset, test_dataset, 64)
    print('> Starting Training Loop:')
    training_loop(model, train_loader, val_loader) 
    print('> Finetuning Workflow Complete.')

if __name__ == '__main__':
    main()

> Loading Model, Tokenizer:
> Loading Dataset:
> Loading Dataset splits:
> Loading DataLoaders:
> Starting Training Loop:


torch.Size([64, 1024])
ids tensor([[ 0, 10,  8,  ...,  8,  7,  2],
        [ 0, 11,  9,  ..., 11, 11,  2],
        [ 0,  9,  9,  ..., 11, 14,  2],
        ...,
        [ 0, 14,  6,  ..., 12,  8,  2],
        [ 0,  4, 16,  ...,  7, 14,  2],
        [ 0,  9,  8,  ..., 11, 12,  2]], device='cuda:0')


NameError: name 'device' is not defined

wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [ ]:
# input_ids = train_data[0]['input_ids']
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# tokenizer = EsmTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')

# rand = torch.rand(input_ids.shape).to(device)
# tokenizer.mask_token_id
# dataset.tokenizer.get_vocab()